In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import sys
from pathlib import Path

# add repo root so swiss_roll_models can be imported from anywhere
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "swiss_roll_models").exists():
        sys.path.insert(0, str(p))
        break

from environment.hilbert_distance import hilbert_analysis as hda

# ============================
# 假设 hilbert_analysis 在同一文件中
# 如果你单独放在 hilbert_module.py 里，就改成：
# from hilbert_module import hilbert_analysis
# ============================

# 你的 hilbert_analysis 已经在上面定义，这里直接用
# class hilbert_analysis: ...

# ============================
# 1. 定义一个简单的 MNIST 模型
# ============================

class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 10)  # 我们主要追踪这一层的权重

    def forward(self, x):
        # x: (B, 1, 28, 28)
        x = x.view(x.size(0), -1)  # 展平
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits






In [ ]:
# ============================
# 2. 训练 + Hilbert 距离追踪
# ============================

def train_mnist_with_hilbert(
    num_epochs=3,
    batch_size=128,
    lr=1e-2,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # ---- 随机种子（方便复现） ----
    torch.manual_seed(42)

    # ---- MNIST 数据 ----
    transform = transforms.Compose([
        transforms.ToTensor(),               # [0, 1]
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    train_dataset = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    # ---- 模型、损失、优化器 ----
    model = MNISTNet().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # ---- 用来存储参数轨迹（Hilbert 分析用）----
    # 我们只追踪最后一层 fc2.weight
    param_traj = []

    def get_positive_flat_last_layer():
        """
        从模型取出最后一层权重 -> 展平 -> 映射到正锥 (abs + eps)
        """
        with torch.no_grad():
            w = model.fc2.weight.detach().clone().view(-1)
            w_pos = w.abs() + 1e-6  # 严格正，避免 hilbert_distance 抛错
        return w_pos

    # 记录初始点
    param_traj.append(get_positive_flat_last_layer())

    # =======================
    # 训练循环
    # =======================
    global_step = 0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            # 统计训练精度（可选）
            running_loss += loss.item() * images.size(0)
            _, predicted = logits.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # 每次参数更新后，记录一次最后一层的正锥权重
            param_traj.append(get_positive_flat_last_layer())
            global_step += 1

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}]  "
              f"Loss: {epoch_loss:.4f}  Acc: {epoch_acc*100:.2f}%")

    print(f"Training finished. Total recorded steps (including init): {len(param_traj)}")

    # =======================
    # 3. Hilbert 距离分析
    # =======================
    # 这里直接把最终权重当作 w_star
    w_star = param_traj[-1]

    # 不做 masking，也不做二次 threshold（因为我们已经 abs+eps 过了）
    hilbert_result = hda.analysis_distance_on_cone(
        param_traj=param_traj,
        w_star=w_star,
        threshold=None,
        ifmask=False,
        if_threshold=False,
        if_self_adaptive=False,
    )

    hilbert_to_final = hilbert_result["hilbert_to_final"]
    hilbert_to_init = hilbert_result["hilbert_to_init"]
    hilbert_between = hilbert_result["hilbert_between"]

    print("\n=== Hilbert 分析结果 (部分) ===")
    print(f"len(hilbert_to_final)  = {len(hilbert_to_final)}  # 每个 step 对 w* 的距离")
    print(f"len(hilbert_to_init)   = {len(hilbert_to_init)}   # 每个 step 对 w0 的距离")
    print(f"len(hilbert_between)   = {len(hilbert_between)}  # 相邻步之间的距离")

    print("\n前 10 步 d_H(w_t, w*):")
    print([float(x) for x in hilbert_to_final[:10]])

    print("\n前 10 步 d_H(w_t, w_0):")
    print([float(x) for x in hilbert_to_init[:10]])

    print("\n前 10 步 d_H(w_t, w_{t-1}):")
    print([float(x) for x in hilbert_between[:10]])

    # 你也可以在这里额外返回结果，方便后续画图 / 做比率分析
    return {
        "model": model,
        "param_traj": param_traj,
        "hilbert_result": hilbert_result,
        "hilbert_to_final": hilbert_to_final,
        "hilbert_to_init": hilbert_to_init,
        "hilbert_between": hilbert_between,
    }


In [ ]:

result=train_mnist_with_hilbert(
    num_epochs=3,
    batch_size=128,
    lr=1e-2,
)

Using device: cuda
Epoch [1/3]  Loss: 0.7633  Acc: 82.10%
Epoch [2/3]  Loss: 0.3615  Acc: 89.85%
Epoch [3/3]  Loss: 0.3080  Acc: 91.19%
Training finished. Total recorded steps (including init): 1408

=== Hilbert 分析结果 (部分) ===
len(hilbert_to_final)  = 1408  # 每个 step 对 w* 的距离
len(hilbert_to_init)   = 1408   # 每个 step 对 w0 的距离
len(hilbert_between)   = 1407  # 相邻步之间的距离

前 10 步 d_H(w_t, w*):
[13.656726837158203, 13.419578552246094, 13.898550033569336, 13.527496337890625, 12.841720581054688, 13.374065399169922, 12.373943328857422, 14.936422348022461, 13.295469284057617, 13.550796508789062]

前 10 步 d_H(w_t, w_0):
[0.0, 2.9013776779174805, 4.150603294372559, 5.575801849365234, 4.823336601257324, 6.833193778991699, 5.641814231872559, 8.132007598876953, 7.649444580078125, 7.533379077911377]

前 10 步 d_H(w_t, w_{t-1}):
[2.9013776779174805, 2.0438406467437744, 3.8183467388153076, 1.483393669128418, 3.898033618927002, 3.74942684173584, 4.775698661804199, 4.3495988845825195, 4.156457424163818, 4.757

In [ ]:
hilbert_to_final=result["hilbert_to_final"]
hilbert_to_init=result["hilbert_to_init"]
hilbert_between=result["hilbert_between"]


In [ ]:
print("\n=== Hilbert 分析结果 (部分) ===")
print(f"len(hilbert_to_final)  = {len(hilbert_to_final)}  # 每个 step 对 w* 的距离")
print(f"len(hilbert_to_init)   = {len(hilbert_to_init)}   # 每个 step 对 w0 的距离")
print(f"len(hilbert_between)   = {len(hilbert_between)}  # 相邻步之间的距离")

print("\n前 10 步 d_H(w_t, w*):")
print([float(x) for x in hilbert_to_final[:10]])

print("\n前 10 步 d_H(w_t, w_0):")
print([float(x) for x in hilbert_to_init[:10]])

print("\n前 10 步 d_H(w_t, w_{t-1}):")
print([float(x) for x in hilbert_between[:10]])

# ============================
# 额外：收缩比率（几何收缩性）
# ============================
ratios_to_final = []
for t in range(len(hilbert_to_final) - 1):
    if hilbert_to_final[t] > 0:
        ratios_to_final.append(hilbert_to_final[t+1] / hilbert_to_final[t])
    else:
        ratios_to_final.append(float("nan"))

ratios_between = []
for t in range(len(hilbert_between) - 1):
    if hilbert_between[t] > 0:
        ratios_between.append(hilbert_between[t+1] / hilbert_between[t])
    else:
        ratios_between.append(float("nan"))

print("\n前 20 个 ratio: d_H(w_{t+1}, w*) / d_H(w_t, w*):")
print(ratios_to_final[:20])

# 简单做一点统计：去掉前几步的剧烈抖动
burn_in = 20
if len(ratios_to_final) > burn_in + 10:
    tail = ratios_to_final[burn_in:]
    tail_clean = [r for r in tail if not math.isnan(r) and r != float("inf")]
    if len(tail_clean) > 0:
        print(f"\n从 step>{burn_in} 之后的 ratio_to_final：")
        print(f"  平均值 ≈ {sum(tail_clean)/len(tail_clean):.4f}")
        print(f"  最小值 ≈ {min(tail_clean):.4f}, 最大值 ≈ {max(tail_clean):.4f}")



=== Hilbert 分析结果 (部分) ===
len(hilbert_to_final)  = 1408  # 每个 step 对 w* 的距离
len(hilbert_to_init)   = 1408   # 每个 step 对 w0 的距离
len(hilbert_between)   = 1407  # 相邻步之间的距离

前 10 步 d_H(w_t, w*):
[13.656726837158203, 13.419578552246094, 13.898550033569336, 13.527496337890625, 12.841720581054688, 13.374065399169922, 12.373943328857422, 14.936422348022461, 13.295469284057617, 13.550796508789062]

前 10 步 d_H(w_t, w_0):
[0.0, 2.9013776779174805, 4.150603294372559, 5.575801849365234, 4.823336601257324, 6.833193778991699, 5.641814231872559, 8.132007598876953, 7.649444580078125, 7.533379077911377]

前 10 步 d_H(w_t, w_{t-1}):
[2.9013776779174805, 2.0438406467437744, 3.8183467388153076, 1.483393669128418, 3.898033618927002, 3.74942684173584, 4.775698661804199, 4.3495988845825195, 4.156457424163818, 4.757666110992432]

前 20 个 ratio: d_H(w_{t+1}, w*) / d_H(w_t, w*):
[0.9826350568668578, 1.035691991328824, 0.9733027046143302, 0.9493050495297438, 1.0414543218531478, 0.9252192926786066, 1.207086694278698

In [ ]:
print("\n最后 200 步的 ratio_to_final：")
print(f"  平均值 ≈ {sum(ratios_tail)/len(ratios_tail):.4f}")
print(f"  最小值 ≈ {min(ratios_tail):.4f}, 最大值 ≈ {max(ratios_tail):.4f}")


In [ ]:
if sys.path and "swiss_roll_models" in sys.path[0]:
    sys.path.pop(0)

# 或者打印一下看看路径情况
print(sys.path)
import matplotlib.pyplot as plt
# ============================
# 画图：Hilbert 距离和收缩比率
# ============================
steps = list(range(len(hilbert_to_final)))

plt.figure()
plt.semilogy(steps, hilbert_to_final)
plt.xlabel("step t")
plt.ylabel("d_H(w_t, w*) (log scale)")
plt.title("Hilbert distance to w* during training")
plt.tight_layout()
plt.savefig("hilbert_to_final.png")

# 为了避免太密，下采样一点再画 ratio
stride = 10
ratio_idx = list(range(0, len(ratios_to_final), stride))
ratio_vals = [ratios_to_final[i] for i in ratio_idx]

plt.figure()
plt.plot(ratio_idx, ratio_vals)
plt.xlabel("step t")
plt.ylabel("ratio d_H(w_{t+1}, w*) / d_H(w_t, w*)")
plt.axhline(1.0, linestyle="--")
plt.title(f"Hilbert contraction ratio (stride={stride})")
plt.tight_layout()
plt.savefig("hilbert_ratio_to_final.png")


['c:\\Users\\ASUS\\Desktop\\cone_dynamics', 'c:\\Users\\ASUS\\Desktop\\cone_dynamics', 'c:\\Users\\ASUS\\Desktop\\cone_dynamics', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\python311.zip', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\DLLs', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311', '', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib\\site-packages', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib\\site-packages\\win32', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib\\site-packages\\Pythonwin']


ModuleNotFoundError: No module named 'matplotlib'